# Lecture 10: Why Does Deep Learning Work?


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons, make_classification, load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
print("Libraries loaded.")


## 1. Properties of the Loss Surface (Ch. 20.3)

The loss surface in deep learning:
- Many **local minima** exist, but most have similar values
- **Saddle points** are more frequent than local minima
- **Flat minima** → better generalization


In [ ]:
# 2D loss surface: effect of flatness on generalization
def sharp_minimum(w1, w2):
    return 5*(w1-1)**2 + 50*(w2-0.5)**2

def flat_minimum(w1, w2):
    return 0.5*(w1-(-1))**2 + 0.8*(w2-(-0.5))**2

w1 = np.linspace(-3, 4, 200)
w2 = np.linspace(-2, 2.5, 200)
W1, W2 = np.meshgrid(w1, w2)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, (name, fn, cx, cy) in zip(axes, [
    ("Sharp Minimum", sharp_minimum, 1.0, 0.5),
    ("Flat Minimum",  flat_minimum,  -1.0, -0.5),
]):
    Z = np.log(fn(W1, W2) + 1)
    ax.contourf(W1, W2, Z, levels=25, cmap='YlOrRd', alpha=0.85)
    ax.contour(W1, W2, Z, levels=25, colors='white', linewidths=0.4, alpha=0.4)
    ax.plot(cx, cy, 'b*', markersize=15, label="Found minimum")
    
    circle = plt.Circle((cx, cy), 0.5, fill=False, color='blue', lw=2, linestyle='--')
    ax.add_patch(circle)
    ax.set_title(f"{name}\n({'sensitive' if 'Sharp' in name else 'robust'} to small perturbations)")
    ax.legend(); ax.grid(alpha=0.2)
    ax.set_xlabel("w1"); ax.set_ylabel("w2")

plt.suptitle("Flat vs Sharp Minima: Effect on Generalization", fontsize=13)
plt.tight_layout()
plt.savefig("fig_nb10_landscape.png", dpi=100, bbox_inches='tight')
plt.show()


## 2. Overparameterization and Generalization (Ch. 20.4)

In classical statistics, too many parameters → overfitting.  
However, in deep learning **very large networks** sometimes generalize better than small ones.

This is called 'implicit regularization': SGD naturally steers toward flat minima in large networks.


In [ ]:
try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    TORCH = True
except ImportError:
    TORCH = False
    print("PyTorch not found; simulated results will be shown.")

if TORCH:
    X_m, y_m = make_moons(n_samples=200, noise=0.2, random_state=42)
    scaler = StandardScaler()
    X_m = scaler.fit_transform(X_m)
    X_tr, X_te, y_tr, y_te = train_test_split(X_m, y_m, test_size=0.3, random_state=42)
    
    Xtr_t = torch.FloatTensor(X_tr)
    ytr_t = torch.LongTensor(y_tr)
    Xte_t = torch.FloatTensor(X_te)
    yte_t = torch.LongTensor(y_te)
    
    def make_net(hidden_size):
        return nn.Sequential(
            nn.Linear(2, hidden_size), nn.ReLU(),
            nn.Linear(hidden_size, hidden_size), nn.ReLU(),
            nn.Linear(hidden_size, 2)
        )
    
    def train_net(net, epochs=500, lr=0.01):
        opt = optim.Adam(net.parameters(), lr=lr)
        criterion = nn.CrossEntropyLoss()
        tr_acc_log, te_acc_log = [], []
        for ep in range(epochs):
            net.train()
            opt.zero_grad()
            out = net(Xtr_t)
            loss = criterion(out, ytr_t)
            loss.backward(); opt.step()
            if ep % 10 == 0:
                net.eval()
                with torch.no_grad():
                    tr_acc = (net(Xtr_t).argmax(1) == ytr_t).float().mean().item()
                    te_acc = (net(Xte_t).argmax(1) == yte_t).float().mean().item()
                    tr_acc_log.append(tr_acc); te_acc_log.append(te_acc)
        return tr_acc_log, te_acc_log
    
    sizes = [4, 16, 64, 256]
    results = {}
    for sz in sizes:
        net = make_net(sz)
        n_params = sum(p.numel() for p in net.parameters())
        tr, te = train_net(net, epochs=600)
        results[sz] = {'tr': tr, 'te': te, 'params': n_params}
        print(f"Hidden={sz:4d} | params={n_params:6d} | final_test_acc={te[-1]:.3f}")
    
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    colors_list = ['steelblue', 'darkorange', 'crimson', 'purple']
    for sz, color in zip(sizes, colors_list):
        ep_range = np.arange(len(results[sz]['tr'])) * 10
        axes[0].plot(ep_range, results[sz]['tr'], color=color, lw=1.5, alpha=0.5)
        axes[0].plot(ep_range, results[sz]['te'], color=color, lw=2,
                    label=f"h={sz} ({results[sz]['params']} params)")
    axes[0].set_title("Train (thin) vs Test (thick) Accuracy")
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Accuracy")
    axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)
    
    param_counts = [results[sz]['params'] for sz in sizes]
    test_accs = [results[sz]['te'][-1] for sz in sizes]
    axes[1].plot(param_counts, test_accs, '-o', color='steelblue', lw=2, markersize=8)
    for sz, pc, ta in zip(sizes, param_counts, test_accs):
        axes[1].annotate(f"h={sz}", (pc, ta), textcoords="offset points", xytext=(5,5))
    axes[1].set_title("Parameter Count vs Test Accuracy")
    axes[1].set_xlabel("Parameter count"); axes[1].set_ylabel("Test accuracy")
    axes[1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig("fig_nb10_overparameterization.png", dpi=100, bbox_inches='tight')
    plt.show()

else:
    print("Simulated results:")
    sizes_sim = [4, 16, 64, 256]
    test_accs_sim = [0.78, 0.88, 0.91, 0.90]
    for s, a in zip(sizes_sim, test_accs_sim): print(f"  h={s}: test_acc={a}")


## 3. Is Depth Really Necessary? (Ch. 20.6)

For the same parameter count: is a deep network better than a shallow one?  
Experiments consistently show the advantage of depth.


In [ ]:
if TORCH:
    X_d, y_d = make_classification(n_samples=500, n_features=10, n_informative=8,
                                    n_redundant=2, random_state=42)
    scaler2 = StandardScaler()
    X_d = scaler2.fit_transform(X_d)
    X_dtr, X_dte, y_dtr, y_dte = train_test_split(X_d, y_d, test_size=0.3)
    Xdtr = torch.FloatTensor(X_dtr); ydtr = torch.LongTensor(y_dtr)
    Xdte = torch.FloatTensor(X_dte); ydte = torch.LongTensor(y_dte)
    
    architectures = {
        "Shallow [10,64,2]":            nn.Sequential(nn.Linear(10,64),nn.ReLU(),nn.Linear(64,2)),
        "Medium  [10,20,20,2]":          nn.Sequential(nn.Linear(10,20),nn.ReLU(),nn.Linear(20,20),nn.ReLU(),nn.Linear(20,2)),
        "Deep    [10,12,12,12,12,2]":    nn.Sequential(nn.Linear(10,12),nn.ReLU(),
                                                       nn.Linear(12,12),nn.ReLU(),
                                                       nn.Linear(12,12),nn.ReLU(),
                                                       nn.Linear(12,12),nn.ReLU(),
                                                       nn.Linear(12,2)),
    }
    
    print(f"{'Architecture':<30} {'Parameters':>12} {'Test Acc':>10}")
    print("-"*55)
    arch_results = {}
    for name, net in architectures.items():
        n_p = sum(p.numel() for p in net.parameters())
        opt_d = optim.Adam(net.parameters(), lr=0.005)
        criterion = nn.CrossEntropyLoss()
        for _ in range(800):
            net.train(); opt_d.zero_grad()
            loss = criterion(net(Xdtr), ydtr)
            loss.backward(); opt_d.step()
        net.eval()
        with torch.no_grad():
            acc = (net(Xdte).argmax(1) == ydte).float().mean().item()
        print(f"{name:<30} {n_p:>12,} {acc:>10.3f}")
        arch_results[name] = acc
    
    fig, ax = plt.subplots(figsize=(8, 4))
    names = list(arch_results.keys())
accs  = list(arch_results.values())
    colors_arch = ['steelblue', 'darkorange', 'crimson']
    bars = ax.bar(names, accs, color=colors_arch, edgecolor='white', width=0.5)
    for bar, acc in zip(bars, accs):
        ax.text(bar.get_x()+bar.get_width()/2, acc+0.005, f"{acc:.3f}", ha='center', fontsize=11)
    ax.set_ylim(0.7, 1.0); ax.set_ylabel("Test Accuracy")
    ax.set_title("Effect of Depth with Similar Parameter Count")
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig("fig_nb10_depth.png", dpi=100, bbox_inches='tight')
    plt.show()

else:
    import matplotlib.pyplot as plt
import numpy as np
    print("PyTorch not found — simulated results:")
    names = ["Shallow [10,64,2]", "Medium [10,20,20,2]", "Deep [10,12,12,12,12,2]"]
    accs  = [0.85, 0.88, 0.91]
    params= [834, 682, 796]
    print(f"{'Architecture':<30} {'Parameters':>12} {'Test Acc':>10}")
    print("-"*55)
    for n, p, a in zip(names, params, accs):
        print(f"{n:<30} {p:>12,} {a:>10.3f}")
    fig, ax = plt.subplots(figsize=(8, 4))
    colors_arch = ['steelblue', 'darkorange', 'crimson']
    bars = ax.bar(names, accs, color=colors_arch, edgecolor='white', width=0.5)
    for bar, acc in zip(bars, accs):
        ax.text(bar.get_x()+bar.get_width()/2, acc+0.005, f"{acc:.3f}", ha='center', fontsize=11)
    ax.set_ylim(0.7, 1.0); ax.set_ylabel("Test Accuracy (simulated)")
    ax.set_title("Effect of Depth with Similar Parameter Count (Simulated)")
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig("fig_nb10_depth_sim.png", dpi=100, bbox_inches='tight')
    plt.show()


## 4. Feature Learning: What Does a DNN Learn?

Let us visualize the hidden layer activations of a deep network → **hierarchical features**


In [ ]:
if TORCH:
    from sklearn.decomposition import PCA
    
    digits = load_digits()
    X_dig, y_dig = digits.data / 16.0, digits.target
    X_dg_tr, X_dg_te, y_dg_tr, y_dg_te = train_test_split(X_dig, y_dig, test_size=0.3, random_state=0)
    
    Xtrain_dg = torch.FloatTensor(X_dg_tr); ytrain_dg = torch.LongTensor(y_dg_tr)
    Xtest_dg  = torch.FloatTensor(X_dg_te); ytest_dg  = torch.LongTensor(y_dg_te)
    
    class DigsNet(nn.Module):
        def __init__(self):
            super().__init__()
            self.fc1 = nn.Linear(64, 128)
            self.fc2 = nn.Linear(128, 64)
            self.fc3 = nn.Linear(64, 32)
            self.fc4 = nn.Linear(32, 10)
            self.relu = nn.ReLU()
        def forward(self, x):
            h1 = self.relu(self.fc1(x))
            h2 = self.relu(self.fc2(h1))
            h3 = self.relu(self.fc3(h2))
            return self.fc4(h3), h1, h2, h3
    
    net_dg = DigsNet()
    opt_dg = optim.Adam(net_dg.parameters(), lr=0.001)
    criterion_dg = nn.CrossEntropyLoss()
    
    for ep in range(300):
        net_dg.train(); opt_dg.zero_grad()
        out, _, _, _ = net_dg(Xtrain_dg)
        loss_dg = criterion_dg(out, ytrain_dg)
        loss_dg.backward(); opt_dg.step()
    
    net_dg.eval()
    with torch.no_grad():
        out_te, h1_te, h2_te, h3_te = net_dg(Xtest_dg)
        acc_dg = (out_te.argmax(1) == ytest_dg).float().mean().item()
    
    print(f"Test accuracy: {acc_dg:.3f}")
    
    layers_act = [X_dg_te, h1_te.numpy(), h2_te.numpy(), h3_te.numpy()]
    layer_names = ["Raw Input (64d)", "Layer 1 (128d)", "Layer 2 (64d)", "Layer 3 (32d)"]
    colors_dig = plt.cm.tab10(np.linspace(0,1,10))
    
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    for ax, acts, name in zip(axes, layers_act, layer_names):
        pca = PCA(n_components=2)
        X_2d = pca.fit_transform(acts)
        for cls in range(10):
            mask = y_dg_te == cls
            ax.scatter(X_2d[mask, 0], X_2d[mask, 1], s=8, alpha=0.6, color=colors_dig[cls], label=str(cls))
        ax.set_title(f"{name}\nPCA 2D")
        ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
    axes[-1].legend(fontsize=7, loc='upper right', title="Digit")
    
    plt.suptitle("Deep Network: Layer-by-Layer Feature Learning (Digits)", fontsize=13)
    plt.tight_layout()
    plt.savefig("fig_nb10_features.png", dpi=100, bbox_inches='tight')
    plt.show()

else:
    from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import numpy as np
    digits = load_digits()
    X_dig, y_dig = digits.data / 16.0, digits.target
    from sklearn.model_selection import train_test_split
    _, X_dg_te, _, y_dg_te = train_test_split(X_dig, y_dig, test_size=0.3, random_state=0)
    print("PyTorch not found — showing PCA of raw input.")
    colors_dig = plt.cm.tab10(np.linspace(0,1,10))
    fig, ax = plt.subplots(figsize=(6, 5))
    pca = PCA(n_components=2)
    X_2d = pca.fit_transform(X_dg_te)
    for cls in range(10):
        mask = y_dg_te == cls
        ax.scatter(X_2d[mask,0], X_2d[mask,1], s=8, alpha=0.6, color=colors_dig[cls], label=str(cls))
    ax.set_title("Raw Digits Data PCA (without PyTorch)")
    ax.legend(fontsize=7, title="Digit")
    plt.tight_layout()
    plt.savefig("fig_nb10_features_pca.png", dpi=100, bbox_inches='tight')
    plt.show()


## 5. Summary: Why Does Deep Learning Work?

| Factor | Description |
|---|---|
| **Loss surface** | Local minima have similar values; SGD steers toward good minima |
| **Implicit regularization** | SGD naturally finds flat minima in large networks |
| **Hierarchical representation** | Deep networks learn abstract features |
| **Overparameterization** | Generalization can improve beyond the interpolation threshold |
| **Depth advantage** | Exponential region growth; richer function space with same parameters |
| **Transfer learning** | Features learned on one task can be reused on others |

---

## Full Course Summary

| Notebook | Topics |
|---|---|
| NB01 | Introduction: Supervised/Unsupervised/RL |
| NB02 | Shallow and Deep Neural Networks, ReLU |
| NB03 | Loss Functions: MSE, BCE, Cross-Entropy |
| NB04 | Training: GD, SGD, Adam, Backprop |
| NB05 | Performance & Regularization |
| NB06 | CNN & ResNet |
| NB07 | Transformers & Attention |
| NB08 | Generative Models: GAN, VAE, Diffusion |
| NB09 | Reinforcement Learning |
| NB10 | Why Does Deep Learning Work? |
